In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from imblearn.over_sampling import SMOTE

In [11]:
def load_data(path):
    df = pd.read_csv(path)
    return df

In [12]:
def data_understanding(df):
    print("=== DATASET SHAPE ===")
    print(df.shape)
    
    print("\n=== MISSING VALUES ===")
    print(df.isnull().sum())
    
    print("\n=== DISTRIBUSI FRAUD ===")
    fraud_count = df['isFraud'].value_counts()
    fraud_percent = df['isFraud'].value_counts(normalize=True) * 100
    
    print("Jumlah:\n", fraud_count)
    print("\nPersentase (%):\n", fraud_percent.round(4))

In [13]:
def clean_data(df):
    df = df.copy()
    
    df.drop(columns=['nameOrig', 'nameDest'], inplace=True, errors='ignore')
    
    if df.isnull().sum().sum() > 0:
        df = df.dropna()
    
    return df

In [14]:
def feature_engineering(df):
    df = df.copy()
    
    df['balance_diff'] = df['oldbalanceOrg'] - df['newbalanceOrig']
    df['balance_ratio'] = df['newbalanceOrig'] / (df['oldbalanceOrg'] + 1)
    df['dest_balance_diff'] = df['oldbalanceDest'] - df['newbalanceDest']
    
    df['is_suspicious'] = (
        (df['oldbalanceOrg'] > 0) & (df['newbalanceOrig'] == 0)
    ).astype(int)
    
    return df

In [15]:
def encode_data(df):
    df = df.copy()
    
    le = LabelEncoder()
    df['type'] = le.fit_transform(df['type'])
    
    return df

In [16]:
def split_data(df):
    X = df.drop('isFraud', axis=1)
    y = df['isFraud']
    
    return train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

In [17]:
def apply_smote(X_train, y_train):
    
    print("\n=== BEFORE SMOTE ===")
    print(y_train.value_counts())
    
    smote = SMOTE(random_state=42)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    
    print("\n=== AFTER SMOTE ===")
    print(pd.Series(y_res).value_counts())
    
    return X_res, y_res

In [18]:
def optimize_memory(X_train, X_test):
    return X_train.astype('float32'), X_test.astype('float32')

In [19]:
def preprocessing_pipeline(path):
    
    # Load
    df = load_data(path)
    
    # Report awal
    data_understanding(df)
    
    # Cleaning
    df = clean_data(df)
    
    # Feature engineering
    df = feature_engineering(df)
    
    # Encoding
    df = encode_data(df)
    
    # Split
    X_train, X_test, y_train, y_test = split_data(df)
    
    # SMOTE (report)
    X_train_res, y_train_res = apply_smote(X_train, y_train)
    
    # Optimize memory
    X_train_res, X_test = optimize_memory(X_train_res, X_test)
    
    return X_train_res, X_test, y_train_res, y_test

In [20]:
X_train, X_test, y_train, y_test = preprocessing_pipeline('paysim.csv')

=== DATASET SHAPE ===
(6362620, 11)

=== MISSING VALUES ===
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

=== DISTRIBUSI FRAUD ===
Jumlah:
 isFraud
0    6354407
1       8213
Name: count, dtype: int64

Persentase (%):
 isFraud
0    99.8709
1     0.1291
Name: proportion, dtype: float64

=== BEFORE SMOTE ===
isFraud
0    5083526
1       6570
Name: count, dtype: int64

=== AFTER SMOTE ===
isFraud
0    5083526
1    5083526
Name: count, dtype: int64


## Modeling

In [21]:
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("\nDistribusi y_train:")
print(y_train.value_counts())

X_train shape: (10167052, 12)
X_test shape : (1272524, 12)

Distribusi y_train:
isFraud
0    5083526
1    5083526
Name: count, dtype: int64


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import matplotlib.pyplot as plt

In [23]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"\n=== {model_name} ===")
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    roc = roc_auc_score(y_test, y_proba)
    print(f"\nROC-AUC: {roc:.4f}")
    
    return roc

In [24]:
lr = LogisticRegression(max_iter=1000, n_jobs=-1)

lr.fit(X_train, y_train)

roc_lr = evaluate_model(lr, X_test, y_test, "Logistic Regression")


=== Logistic Regression ===

Confusion Matrix:
[[1164873  106008]
 [     49    1594]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.92      0.96   1270881
           1       0.01      0.97      0.03      1643

    accuracy                           0.92   1272524
   macro avg       0.51      0.94      0.49   1272524
weighted avg       1.00      0.92      0.96   1272524


ROC-AUC: 0.9897


In [ ]:
#threshold tuning
y_proba = lr.predict_proba(X_test)[:, 1]

# ubah threshold (coba 0.9 dulu)
threshold = 0.9
y_pred_new = (y_proba > threshold).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_new))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_new))

Confusion Matrix:
[[1262936    7945]
 [    354    1289]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00   1270881
           1       0.14      0.78      0.24      1643

    accuracy                           0.99   1272524
   macro avg       0.57      0.89      0.62   1272524
weighted avg       1.00      0.99      1.00   1272524



In [27]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

roc_rf = evaluate_model(rf, X_test, y_test, "Random Forest")


=== Random Forest ===

Confusion Matrix:
[[1259479   11402]
 [     17    1626]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00   1270881
           1       0.12      0.99      0.22      1643

    accuracy                           0.99   1272524
   macro avg       0.56      0.99      0.61   1272524
weighted avg       1.00      0.99      0.99   1272524


ROC-AUC: 0.9996


In [28]:
y_proba_rf = rf.predict_proba(X_test)[:, 1]

threshold = 0.9
y_pred_rf_new = (y_proba_rf > threshold).astype(int)

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_rf_new))
print(classification_report(y_test, y_pred_rf_new))

[[1269446    1435]
 [     91    1552]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.52      0.94      0.67      1643

    accuracy                           1.00   1272524
   macro avg       0.76      0.97      0.83   1272524
weighted avg       1.00      1.00      1.00   1272524

